# Task 8: U-Net semantic segmentation with custom Dice + BCE loss

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [2]:
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.ReLU(),
        nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.ReLU()
    )

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = conv_block(3, 16)
        self.enc2 = conv_block(16, 32)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = conv_block(32, 64)
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = conv_block(64, 32)
        self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = conv_block(32, 16)
        self.out = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.out(d1))


In [3]:
def dice_loss(pred, target, eps=1e-6):
    pred = pred.reshape(pred.size(0), -1)
    target = target.reshape(target.size(0), -1)
    inter = (pred*target).sum(1)
    union = pred.sum(1) + target.sum(1)
    return 1 - ((2*inter+eps)/(union+eps)).mean()

def combined_loss(pred, target):
    return F.binary_cross_entropy(pred, target) + dice_loss(pred, target)


In [4]:
model = UNet()
x = torch.randn(2, 3, 64, 64)
mask = (torch.rand(2, 1, 64, 64) > 0.5).float()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    opt.zero_grad()
    pred = model(x)
    loss = combined_loss(pred, mask)
    loss.backward()
    opt.step()
    print(epoch, loss.item())


0 1.2147397994995117
1 1.2120155096054077
2 1.2086453437805176
3 1.2044425010681152
4 1.1993787288665771
